# 🚀 Pipeline ETL Adquirentes - Google Colab

**Versão**: 1.3.0  
**Última atualização**: 2026-06-04

Este notebook executa o pipeline completo de ETL para processamento de dados das adquirentes Cielo e Pagar.me.

✨ **Novo**: Paths parametrizáveis e suporte a múltiplas extensões (.csv, .xls, .xlsx)

---

## 📋 Pré-requisitos

1. Arquivos de entrada no Google Drive
2. Permissões de escrita no Drive
3. Código do projeto (via upload ou GitHub)

---

## 1️⃣ Setup Inicial

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive montado com sucesso!")

In [ ]:
# Instalar dependências
!pip install pandas openpyxl -q

import pandas as pd
print(f"✓ Pandas {pd.__version__} instalado")
print("✓ Openpyxl instalado")

## 2️⃣ Configuração de Paths (IMPORTANTE!) 🆕

### ⚙️ Configure seus caminhos personalizados aqui

**Atenção**: Ajuste os paths abaixo conforme a estrutura do seu Google Drive

In [ ]:
# ========================================
# CONFIGURAÇÃO DE PATHS PERSONALIZADOS
# ========================================

# Opção 1: Shared drives (Drives compartilhados)
BASE_DRIVE = "/content/drive/Shared drives/Revenue Assurance/Dash/"

# Opção 2: MyDrive (Drive pessoal)
# BASE_DRIVE = "/content/drive/MyDrive/GRAN/Revenue Assurance/Dash/"

# Opção 3: Caminho completamente customizado
# BASE_DRIVE = "/content/drive/MyDrive/SEU_CAMINHO/"

# Paths específicos (baseados no BASE_DRIVE)
CUSTOM_PATHS = {
    'input': BASE_DRIVE + "Backend Adq/",
    'temp': BASE_DRIVE + "Temp Adq/",
    'output': BASE_DRIVE + "AGREGADO CONSOLIDADO/"
}

# Google Sheets (opcional - configure se for diferente)
CUSTOM_SHEETS = {
    'spreadsheet_id': "1PdRMOoitt0HMXnPbU0Jsd3BVU3u8OjWNESWvHIYta4A",
    'sheet_name': "Adquirentes"
}

# Mostrar configuração
print("📂 Configuração de Paths:")
print(f"  Input:  {CUSTOM_PATHS['input']}")
print(f"  Temp:   {CUSTOM_PATHS['temp']}")
print(f"  Output: {CUSTOM_PATHS['output']}")
print(f"\n📊 Google Sheets:")
print(f"  ID:     {CUSTOM_SHEETS['spreadsheet_id']}")
print(f"  Aba:    {CUSTOM_SHEETS['sheet_name']}")

In [ ]:
# Verificar se os diretórios existem
import os

print("🔍 Verificando paths...\n")

for nome, path in CUSTOM_PATHS.items():
    existe = os.path.exists(path)
    status = "✓" if existe else "✗"
    print(f"{status} {nome}: {path}")
    
    if not existe:
        print(f"   ⚠️  Diretório não encontrado!")
        criar = input(f"   Criar diretório? (s/N): ")
        if criar.lower() == 's':
            os.makedirs(path, exist_ok=True)
            print(f"   ✓ Diretório criado")

print("\n✓ Verificação concluída")

## 3️⃣ Obter Código do Projeto

### Opção A: Upload Manual

In [ ]:
# Fazer upload do arquivo etl_adquirentes.zip
from google.colab import files

print("📤 Faça upload do arquivo etl_adquirentes.zip")
uploaded = files.upload()

# Extrair
!unzip -q etl_adquirentes.zip
print("✓ Arquivos extraídos")

### Opção B: Clone do GitHub

In [ ]:
# Clone do repositório GitHub
# IMPORTANTE: Substitua SEU_USUARIO pelo seu usuário GitHub

!git clone https://github.com/SEU_USUARIO/etl-adquirentes.git etl_adquirentes
print("✓ Repositório clonado")

## 4️⃣ Aplicar Configuração Personalizada

In [ ]:
# Navegar para o diretório do projeto
%cd /content/etl_adquirentes

# Adicionar ao PYTHONPATH
import sys
sys.path.insert(0, '/content/etl_adquirentes')

print("✓ Diretório configurado")

In [ ]:
# Aplicar configuração personalizada de paths
from config import settings

# Sobrescrever com paths personalizados
settings.PATHS = CUSTOM_PATHS
settings.GOOGLE_SHEETS = CUSTOM_SHEETS

print("✓ Configuração personalizada aplicada")
print(f"\n📂 Paths ativos:")
for key, value in settings.PATHS.items():
    print(f"  {key}: {value}")

## 5️⃣ Verificar Arquivos de Entrada

In [ ]:
# Listar arquivos de entrada
from utils.leitura_arquivos import listar_arquivos_por_extensao

print("📁 Arquivos de entrada:")
arquivos = listar_arquivos_por_extensao(CUSTOM_PATHS['input'])

for ext, lista in arquivos.items():
    print(f"\n  {ext}: {len(lista)} arquivo(s)")
    for arq in lista[:5]:  # Mostrar primeiros 5
        print(f"    - {arq}")
    if len(lista) > 5:
        print(f"    ... e mais {len(lista) - 5} arquivo(s)")

total = sum(len(lista) for lista in arquivos.values())
print(f"\n✓ Total: {total} arquivos encontrados")

## 6️⃣ Executar Pipeline

### Opção A: Pipeline Completo (21 scripts)

In [ ]:
# Executar pipeline completo
# Tempo estimado: 5-15 minutos

!python orchestrator.py

### Opção B: Pipeline Parcial

In [ ]:
# Apenas Cielo (scripts 01-09)
!python run_cielo.py

In [ ]:
# Apenas Pagar.me (scripts 10-18)
!python run_pagarme.py

### Opção C: Scripts Individuais

In [ ]:
# Exemplo: Apenas consolidação final
!python scripts/final/19_consolida_final.py

In [ ]:
# Exemplo: Apenas formatação brasileira
!python scripts/final/21_formatar_brasileiro.py

In [ ]:
# Exemplo: Apenas envio ao Google Sheets
!python scripts/final/20_envia_sheets.py

## 7️⃣ Verificar Resultados

In [ ]:
# Listar arquivos de saída
print("📊 Arquivos consolidados gerados:")
!ls -lh "{CUSTOM_PATHS['output']}"

In [ ]:
# Visualizar primeiras linhas do arquivo formatado
import pandas as pd

arquivo = CUSTOM_PATHS['output'] + "AGREGADO_CONSOLIDADO_BR.csv"

try:
    df = pd.read_csv(arquivo, sep=";", nrows=10)
    print("✓ Arquivo formatado (primeiras 10 linhas):")
    print(df)
except FileNotFoundError:
    print("⚠ Arquivo ainda não foi gerado. Execute o pipeline primeiro.")

In [ ]:
# Estatísticas rápidas
try:
    df_full = pd.read_csv(arquivo, sep=";")
    
    print(f"📈 Estatísticas:")
    print(f"  Total de registros: {len(df_full)}")
    print(f"\n  Registros por Status:")
    print(df_full['Status'].value_counts())
    print(f"\n  Registros por EC:")
    print(df_full['EC'].value_counts())
    print(f"\n  Registros por Adquirente:")
    print(df_full['Adquirente'].value_counts())
except:
    print("⚠ Não foi possível carregar estatísticas")

## 8️⃣ Download de Resultados (Opcional)

In [ ]:
# Download do arquivo formatado
from google.colab import files

arquivo_download = CUSTOM_PATHS['output'] + "AGREGADO_CONSOLIDADO_BR.csv"

try:
    files.download(arquivo_download)
    print("✓ Download iniciado")
except:
    print("⚠ Erro ao fazer download. Verifique se o arquivo existe.")

## 9️⃣ Testes (Opcional)

In [ ]:
# Testar funções de formatação
from utils.normalizacao import formatar_data_br, formatar_moeda_br

# Testar data
data_teste = formatar_data_br('2026-06-04')
print(f"Teste Data: '2026-06-04' → '{data_teste}'")
assert data_teste == '04/06/2026', "❌ Formatação de data falhou!"
print("✓ Formatação de data OK")

# Testar moeda
moeda_teste = formatar_moeda_br(1234.56)
print(f"Teste Moeda: 1234.56 → '{moeda_teste}'")
assert moeda_teste == 'R$ 1.234,56', "❌ Formatação de moeda falhou!"
print("✓ Formatação de moeda OK")

print("\n🎉 Todos os testes passaram!")

In [ ]:
# Testar leitura de arquivos com múltiplas extensões
from utils.leitura_arquivos import detectar_e_ler_arquivo

# Teste de detecção automática
print("🧪 Testando detecção automática de extensões...")

df_teste, ext = detectar_e_ler_arquivo(
    CUSTOM_PATHS['input'],
    'Historico-de-vendas-pre_1',  # Sem extensão
    ['.xlsx', '.xls', '.csv'],
    skiprows=9,
    nrows=2  # Ler apenas 2 linhas para teste
)

if df_teste is not None:
    print(f"✓ Arquivo encontrado com extensão: {ext}")
    print(f"  Colunas: {list(df_teste.columns)}")
    print(f"  Linhas lidas: {len(df_teste)}")
else:
    print("⚠ Arquivo não encontrado (teste pulado)")

## 🔧 Utilitários

### Reconfigurar Paths

In [ ]:
# Se precisar mudar os paths após inicialização
from config import settings

# Opção: Configurar manualmente
settings.PATHS['input'] = "/content/drive/MyDrive/NOVO_CAMINHO/"
settings.PATHS['temp'] = "/content/drive/MyDrive/TEMP/"
settings.PATHS['output'] = "/content/drive/MyDrive/OUTPUT/"

print("✓ Paths reconfigurados")
print(settings.PATHS)

### Limpar Cache

In [ ]:
# Limpar arquivos __pycache__
!find . -type d -name '__pycache__' -exec rm -rf {} + 2>/dev/null
print("✓ Cache limpo")

### Reautenticar Google Drive

In [ ]:
# Se houver problemas de permissão
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Drive remontado")

---

## 📚 Novidades v1.3.0

### ✨ Paths Parametrizáveis
- Configure seus próprios caminhos do Google Drive
- Suporte para `MyDrive` ou `Shared drives`
- Validação automática de diretórios

### ✨ Múltiplas Extensões
- Detecta automaticamente: `.xlsx`, `.xls`, `.csv`
- Leitura inteligente baseada na extensão
- Funções utilities para leitura/escrita

---

## 📚 Documentação

- **README.md**: Documentação completa do projeto
- **README_COLAB.md**: Guia rápido para Colab
- **GUIA_GOOGLE_COLAB.md**: Guia detalhado de uso no Colab
- **FORMATACAO_BRASILEIRA.md**: Guia de formatação pt_BR
- **CHANGELOG.md**: Histórico de versões

---

## 🆘 Problemas Comuns

### Erro: ModuleNotFoundError
```python
import sys
sys.path.insert(0, '/content/etl_adquirentes')
```

### Erro: Arquivo não encontrado
```python
# Verificar paths
from config import settings
print(settings.PATHS)

# Reconfigar se necessário (ver célula "Reconfigurar Paths")
```

### Timeout de Sessão
- Mantenha a aba ativa
- Execute em partes menores
- Use GPU runtime (Runtime > Change runtime type)

---

**✅ Pipeline ETL Adquirentes v1.3.0**  
**🇧🇷 Com formatação brasileira (DD/MM/YYYY, R$ 1.234,56)**  
**⚙️ Paths parametrizáveis**  
**📄 Suporte a .csv, .xls, .xlsx**
